# 하이브리드DB_실습 — 정형+JSONB+벡터 한 스키마에 담기

이 노트북은 `하이브리드DB_개인과제.md`의 R1~R3(필수)·도전1·도전2·보너스가 참조하는 실습 노트북입니다.

- JSONB에서 만든 `documents` 테이블(`title`, `metadata` JSONB, `company`)을 그대로 이어받아 확장합니다.
- pgvector에서 증명된 대로 **이 환경은 실제 `CREATE EXTENSION vector`·`VECTOR` 타입·HNSW 인덱스가 100% 동작**합니다 — 브루트포스 대체 없이 진짜로 만듭니다.
- 벡터 값 자체는 `pgvector_심화.md`의 `pgvec_bench`와 같은 기법(`doc_type`별 중심 벡터 + 노이즈)으로 생성한 **합성 벡터**입니다. 이 과제의 핵심은 "의미가 얼마나 정교하게 검색되는가"가 아니라 **"정형+JSONB+벡터를 한 스키마·한 쿼리로 결합하는 인덱스·SQL 설계"** 이기 때문에, 실제 BGE-M3 임베딩 대신 규모를 자유롭게 조절할 수 있는 합성 벡터를 씁니다 (실제 문장 임베딩은 8교시에서 이미 충분히 다뤘습니다).
- 이 노트북은 **재실행해도 안전**합니다(멱등) — 이미 컬럼·인덱스·데이터가 있으면 건너뛰고, 없으면 만듭니다.

---
## Step 0 — 연결 확인 및 `documents` 테이블 준비 (멱등)

In [1]:
# 📖 문법 참고: import os, time, json 처럼 쉼표로 나열하면 여러 모듈을 한 줄에 불러올 수 있습니다.
#    os(환경변수·경로), time(시간 측정 — 이 노트북 뒤에서 필요할 때 바로 쓸 수 있도록 미리 준비),
#    json(파이썬 값 <-> JSON 문자열 변환)은 모두 파이썬 표준 라이브러리라 따로 설치할 필요가 없습니다.
import os, time, json
import numpy as np      # numpy: 대량의 숫자 배열(벡터·행렬) 연산 라이브러리. np는 관례적으로 쓰는 별칭.
import psycopg           # psycopg: 파이썬에서 PostgreSQL에 접속·쿼리하기 위한 드라이버(v3)
from dotenv import load_dotenv
# register_vector: pgvector가 추가하는 VECTOR 타입을 이 연결에서 numpy 배열과 자동으로 주고받게
# 등록해주는 함수 — 아래 register_vector(conn)에서 실제로 적용합니다.
from pgvector.psycopg import register_vector

load_dotenv()   # 같은 폴더의 .env 파일을 읽어 KEY=VALUE들을 환경변수로 등록합니다.

# os.environ.get("PGPASSWORD", "postgres"): 환경변수 PGPASSWORD가 있으면 그 값을, 없으면 기본값
# "postgres"를 돌려줍니다 — os.environ["PGPASSWORD"]와 달리 값이 없어도 에러 없이 넘어갑니다.
pw = os.environ.get("PGPASSWORD", "postgres")
conn = psycopg.connect(host="localhost", port=5432, dbname="course_db", user="postgres", password=pw)

# conn.execute(...): psycopg v3는 커서를 따로 안 만들어도 연결(conn)에서 바로 execute를 부를 수
# 있습니다(내부적으로 임시 커서를 하나 만들어 실행). 결과가 필요 없는 DDL(구조를 바꾸는 명령)을
# 실행할 때 특히 간결합니다.
# CREATE EXTENSION IF NOT EXISTS vector: 이 데이터베이스에서 pgvector 확장(VECTOR 타입·벡터 연산자·
# HNSW 인덱스)을 켭니다. IF NOT EXISTS 덕분에 이미 켜져 있어도 에러 없이 넘어갑니다(재실행 안전).
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
conn.commit()          # 방금 켠 확장을 데이터베이스에 확정합니다.
register_vector(conn)  # 이 연결에서 VECTOR 컬럼 값 <-> numpy 배열 자동 변환을 활성화합니다.

# conn.info.dbname: 현재 연결이 물려있는 데이터베이스 이름을 속성으로 바로 조회합니다.
# conn.execute(...).fetchone()[0]: "실행 결과의 첫 행, 첫 컬럼 값 하나만" 바로 꺼내는 흔한 패턴 —
# fetchone()이 첫 행을 튜플로 돌려주고 [0]으로 그 튜플의 첫 값을 꺼냅니다.
print("연결 완료 ->", conn.info.dbname, "| pgvector:", conn.execute("SELECT extversion FROM pg_extension WHERE extname='vector'").fetchone()[0])

연결 완료 -> course_db | pgvector: 0.8.6


In [2]:
# documents 테이블이 없으면 최소 골격으로 생성, 있으면 그대로 사용 (9/17 JSONB 실습과 연결)
# SERIAL PRIMARY KEY: 값을 안 줘도 1,2,3...으로 자동 증가하는 기본키. TEXT: 길이 제한 없는 문자열.
# JSONB: PostgreSQL이 이진 형태로 저장·색인하는 JSON 타입(9/17에서 다룬 것과 동일).
conn.execute("CREATE TABLE IF NOT EXISTS documents (id SERIAL PRIMARY KEY, title TEXT, metadata JSONB)")
conn.commit()

n_before = conn.execute("SELECT count(*) FROM documents").fetchone()[0]
print(f"documents 현재 건수: {n_before}건")   # f-string: 문자열 안 {}에 변수 값을 그대로 끼워 넣는 문법

if n_before == 0:
    # 처음 실행하는 환경이라면 9/17 JSONB 실습과 동일한 4건을 먼저 넣어 연결을 유지합니다.
    # 📖 문법 참고: seed4는 (제목 문자열, 메타데이터 딕셔너리) 쌍 4개를 담은 파이썬 리스트입니다.
    #    딕셔너리 값으로 리스트("tags": [...])나 중첩 딕셔너리("info": {...})도 자유롭게 넣을 수 있는데,
    #    이것이 그대로 JSONB 컬럼에 저장되어 관계형 테이블 한 칸에 유연한 구조를 담을 수 있게 해줍니다.
    seed4 = [
        ("삼성 사업보고서", {"tags": ["공시", "연간"], "author": "김영수", "company": "삼성", "doc_type": "사업보고서"}),
        ("LG 감사보고서",   {"author": "이미영", "company": "LG", "doc_type": "감사보고서", "financial_year": 2025}),
        ("현대 반기보고서", {"tags": ["공시", "반기"], "pages": 235, "author": "박지수", "company": "현대", "doc_type": "반기보고서"}),
        ("복잡한 문서",     {"info": {"author": {"dept": "재무", "name": "홍길동"}}}),
    ]
    # with conn.cursor() as cur: "with"는 파이썬의 컨텍스트 매니저 문법 — 블록이 끝나면 cur가 자동으로
    # 닫힙니다(닫는 코드를 직접 안 써도 됨). cur.executemany(sql, 리스트)는 같은 SQL을 리스트의 각
    # 튜플에 대해 반복 실행합니다 — for문으로 execute를 4번 부르는 것과 같은 효과를 한 번에 냅니다.
    # [(t, psycopg.types.json.Json(m)) for t, m in seed4]: 리스트 컴프리헨션 — seed4의 각 (t, m)
    # 쌍을 풀어서, m(파이썬 딕셔너리)을 psycopg.types.json.Json()으로 감싸 새 튜플을 만든 리스트입니다.
    # Json()으로 감싸야 psycopg가 이 값을 JSONB로 인식해 안전하게 저장합니다(그냥 딕셔너리를 넘기면 에러).
    with conn.cursor() as cur:
        cur.executemany("INSERT INTO documents (title, metadata) VALUES (%s, %s)",
                         [(t, psycopg.types.json.Json(m)) for t, m in seed4])
    conn.commit()
    print("초기 4건(9/17 JSONB 실습과 동일) 삽입 완료")
else:
    print("기존 데이터 유지 -> 그대로 확장합니다")

documents 현재 건수: 4건
기존 데이터 유지 -> 그대로 확장합니다


---
## 구현 — R1·R2·R3

📓 이 노트북이 `documents` 테이블을 대상으로 R1(컬럼 승격)·R2(인덱스 4종)·R3(`hybrid_search()`)를 순서대로 구현합니다.

### R1 · 컬럼 승격 + 합성 데이터로 규모 확장

In [3]:
# R1-a — 컬럼 추가 (NULL 허용, 재실행 안전)
# 📖 문법 참고: ALTER TABLE ... ADD COLUMN IF NOT EXISTS는 이미 있는 테이블에 새 컬럼을 추가하는
#    명령입니다. IF NOT EXISTS 덕분에 이미 그 컬럼이 있어도 에러 없이 넘어갑니다(재실행 안전).
#    이 5줄은 세미콜론(;)으로 구분된 별개의 SQL문인데, conn.execute()에 파라미터(%s) 없이 한
#    문자열로 넘기면 psycopg가 순서대로 전부 실행해줍니다 — execute를 5번 따로 부를 필요가 없습니다.
# 각 컬럼 타입: VARCHAR(100)/VARCHAR(50)은 최대 길이가 있는 문자열, SMALLINT는 작은 범위의 정수,
# TEXT는 길이 제한 없는 문자열, VECTOR(1024)는 pgvector 확장이 제공하는 "1024차원 숫자 벡터" 타입입니다
# (임베딩처럼 숫자 배열을 통째로 한 컬럼에 저장하고, 벡터끼리 거리를 계산할 수 있게 해줍니다).
conn.execute("""
ALTER TABLE documents ADD COLUMN IF NOT EXISTS company VARCHAR(100);
ALTER TABLE documents ADD COLUMN IF NOT EXISTS doc_type VARCHAR(50);
ALTER TABLE documents ADD COLUMN IF NOT EXISTS financial_year SMALLINT;
ALTER TABLE documents ADD COLUMN IF NOT EXISTS content TEXT;
ALTER TABLE documents ADD COLUMN IF NOT EXISTS embedding VECTOR(1024);
""")
conn.commit()
print("컬럼 준비 완료")

컬럼 준비 완료


In [4]:
# R1-b — 백필(컬럼마다 독립적으로 COALESCE — 이미 채워진 컬럼은 보존, 비어있는 컬럼만 채움 -> 재실행 안전)
# (수정: 원래 "WHERE doc_type IS NULL"로 세 컬럼을 한꺼번에 게이팅했으나, 9/17에 company만
#  (또는 company·doc_type까지) 이미 승격한 사람은 그 시점에 doc_type이 NOT NULL이라 이 행 전체가
#  걸러져 financial_year가 조용히 안 채워지는 문제가 있었다. 컬럼별 COALESCE로 대체.)
# 📖 문법 참고: COALESCE(a, b)는 a가 NULL이면 b를, 아니면 a를 그대로 돌려줍니다 — 여기서는
#    "컬럼에 이미 값이 있으면 그대로 두고, 없을 때만 metadata의 값으로 채운다"는 뜻입니다.
#    ->>'키'는 JSONB에서 그 키의 값을 텍스트로 꺼내는 연산자입니다(→ ->는 JSONB로, ->>는 텍스트로
#    꺼낸다는 차이가 있습니다). financial_year는 SMALLINT 컬럼이라 텍스트로 꺼낸 값을 다시
#    ::SMALLINT로 캐스팅해서 넣습니다. UPDATE ... WHERE 조건이 있으므로, 이미 세 컬럼이 모두 채워진
#    행은 아예 건드리지 않습니다(재실행해도 불필요한 쓰기가 없음).
conn.execute("""
UPDATE documents SET
    company        = COALESCE(company, metadata->>'company'),
    doc_type       = COALESCE(doc_type, metadata->>'doc_type'),
    financial_year = COALESCE(financial_year, (metadata->>'financial_year')::SMALLINT)
WHERE company IS NULL OR doc_type IS NULL OR financial_year IS NULL;
""")
conn.commit()

# R1-c — 백필 정확성 검증 (IS DISTINCT FROM: NULL도 정확히 비교)
# metadata에 해당 키가 "현재" 남아있을 때만 비교한다 — 9/17 도전2의 선택 항목(JSONB에서 제거)까지
# 마친 사람은 컬럼엔 값이 있지만 metadata엔 키가 없는 것이 정상이므로, 그 경우를 오탐하지 않기 위함.
# 📖 문법 참고: metadata ? '키'는 JSONB 안에 그 키가 존재하는지(값이 NULL이어도 "존재는 한다"이면
#    참) 확인하는 연산자입니다. IS DISTINCT FROM은 "!="(다르다)와 거의 같지만, 둘 중 하나가 NULL일
#    때도 안전하게 참/거짓을 판단합니다 — 보통의 !=는 NULL과 비교하면 결과가 항상 NULL(모름)이
#    되어버려 원하는 대로 걸러지지 않는데, IS DISTINCT FROM은 그런 경우도 명확히 "다르다"로 처리합니다.
mismatch = conn.execute("""
SELECT count(*) FROM documents
WHERE (metadata ? 'company'        AND company        IS DISTINCT FROM metadata->>'company')
   OR (metadata ? 'doc_type'       AND doc_type       IS DISTINCT FROM metadata->>'doc_type')
   OR (metadata ? 'financial_year' AND financial_year IS DISTINCT FROM (metadata->>'financial_year')::SMALLINT);
""").fetchone()[0]
print(f"[검증] 컬럼 값 vs JSONB 원본 불일치 건수: {mismatch}건  (0이어야 통과)")
# assert 조건, 메시지: 조건이 거짓이면 그 자리에서 AssertionError를 일으켜 실행을 멈춥니다 —
# "여기까지는 반드시 참이어야 한다"는 것을 코드로 강제하는 파이썬 문법입니다.
assert mismatch == 0, "백필 불일치 발견 -> 마이그레이션 순서를 다시 확인하세요"

# for r in ....fetchall(): fetchall()은 조회 결과 전체를 튜플의 리스트로 돌려주고,
# for문으로 한 행(r)씩 꺼내 출력합니다(앞의 fetchone()이 "한 행만"인 것과 대비됩니다).
for r in conn.execute("SELECT id, title, company, doc_type, financial_year FROM documents ORDER BY id LIMIT 4").fetchall():
    print(r)

[검증] 컬럼 값 vs JSONB 원본 불일치 건수: 0건  (0이어야 통과)
(1, '삼성 사업보고서', '삼성', '사업보고서', None)
(2, 'LG 감사보고서', 'LG', '감사보고서', 2025)
(3, '현대 반기보고서', '현대', '반기보고서', None)
(4, '복잡한 문서', None, None, None)


In [5]:
# R1-d — 합성 데이터로 규모 확장 (300건까지 top-up, 이미 300건 이상이면 건너뜀 -> 재실행 안전)
# 벡터는 9/22 pgvector_심화.md의 pgvec_bench와 같은 기법: doc_type별 "중심 벡터" + 노이즈
# -> 같은 doc_type끼리는 벡터가 가깝고 다른 doc_type과는 멀어짐 (실제 임베딩의 군집 성질을 흉내)
TARGET_N = 300   # 최종적으로 맞출 목표 총 건수
DIM = 1024       # 벡터 차원 수 — 위 R1-a에서 만든 VECTOR(1024) 컬럼과 반드시 같아야 함

# 아래 리스트·딕셔너리들은 합성 데이터를 만들 때 무작위로 뽑아 쓸 "재료 풀(pool)"입니다.
# 이 자체는 실행되는 로직이 아니라 데이터일 뿐이라 SQL·특별한 문법은 없습니다.
COMPANIES = ["삼성", "LG", "현대", "SK", "포스코", "신한", "KB", "한화"]
DOC_TYPES = ["사업보고서", "감사보고서", "반기보고서", "분기보고서", "계약서", "press", "정관", "이사회의사록"]
YEARS = [2022, 2023, 2024, 2025]
TAGS_POOL = ["공시", "연간", "반기", "분기", "대외비", "초안", "최종본", "긴급", "검토필요"]
AUTHORS = ["김영수", "이미영", "박지수", "정하늘", "홍길동", "최수민", "강다은"]
CONTRACT_PHRASES = [
    "본 계약서는 손해배상 조항을 포함한다.",
    "계약 해지 시 손해배상 책임 범위를 별도로 정한다.",
    "납품 지연에 따른 손해배상액은 계약금의 10%로 한다.",
]
# doc_type별로 넣을 기본 문장 — 아래에서 GENERIC_PHRASES.get(doc_type, 기본값) 형태로 조회합니다.
GENERIC_PHRASES = {
    "사업보고서": "해당 연도 매출 및 손익 현황을 요약한다.", "감사보고서": "외부감사인의 재무제표 적정 의견을 포함한다.",
    "반기보고서": "상반기 실적과 하반기 전망을 다룬다.", "분기보고서": "분기별 매출·영업이익 변동을 설명한다.",
    "press": "회사의 주요 경영 활동을 대외에 공표한다.", "정관": "회사의 목적과 조직 운영 원칙을 규정한다.",
    "이사회의사록": "이사회 결의 사항과 참석 현황을 기록한다.",
}

n_now = conn.execute("SELECT count(*) FROM documents").fetchone()[0]
n_add = TARGET_N - n_now   # 목표까지 몇 건이 모자란지 (이미 300건 이상이면 0 이하가 됨)
if n_add <= 0:
    print(f"이미 {n_now}건 -> 추가 생성 없이 건너뜀")
else:
    # 📖 문법 참고: np.random.default_rng(7)은 numpy의 "난수 생성기"를 만드는데, 시드값 7을 고정했기
    #    때문에 이 셀을 다시 실행해도 매번 같은 순서의 "난수"가 나옵니다(재현성 확보). 아래 rng.normal·
    #    rng.integers·rng.choice는 모두 이 생성기에서 난수를 뽑는 메서드들입니다.
    rng = np.random.default_rng(7)
    # rng.normal(size=(행, 열)): 정규분포를 따르는 난수를 지정한 모양(shape)의 배열로 한 번에 생성합니다.
    # 여기서는 "doc_type 개수 x 1024차원" 행렬 — doc_type마다 하나씩 "중심 벡터"를 무작위로 만드는 것.
    # .astype(np.float32): 값의 타입을 32비트 실수로 맞춥니다(임베딩·pgvector에서 흔히 쓰는 정밀도).
    centers = rng.normal(size=(len(DOC_TYPES), DIM)).astype(np.float32)
    # np.linalg.norm(centers, axis=1, keepdims=True): 각 행(벡터)의 길이(노름)를 계산합니다.
    # axis=1은 "행 방향으로" 계산하라는 뜻(전체를 한 숫자로 뭉개지 않고 벡터별로 하나씩), keepdims=True는
    # 결과 모양을 나눗셈이 그대로 되도록 (N,1) 형태로 유지합니다. centers /= ...는 "각 벡터를 자기
    # 길이로 나눠 길이를 1로 만든다(단위벡터화)"는 뜻 — 코사인 거리 계산을 쓸 것이므로 방향만 남깁니다.
    centers /= np.linalg.norm(centers, axis=1, keepdims=True)

    rows = []   # 아래 반복문에서 만든 (title, content, ...) 튜플들을 차곡차곡 담을 빈 리스트
    for _ in range(n_add):
        # for _ in range(n): 반복 횟수만 필요하고 반복 변수 자체는 안 쓸 때 관례적으로 밑줄(_)을 씁니다.
        # rng.integers(0, n): 0 이상 n 미만의 정수 난수 하나. 세미콜론(;)으로 한 줄에 두 문장을 이어 쓸 수 있습니다.
        dt_idx = rng.integers(0, len(DOC_TYPES)); doc_type = DOC_TYPES[dt_idx]
        company = COMPANIES[rng.integers(0, len(COMPANIES))]
        year = int(YEARS[rng.integers(0, len(YEARS))])   # numpy 정수 타입을 파이썬 기본 int로 변환
        author = AUTHORS[rng.integers(0, len(AUTHORS))]
        pages = int(rng.integers(20, 300))
        k_tags = rng.integers(0, 3)   # 태그를 0~2개 붙이기로 결정 (3은 포함 안 됨)
        # 조건부 표현식(삼항 연산자와 비슷): "만약 k_tags > 0이면 앞의 식, 아니면 else 뒤의 값"을
        # 한 줄로 씁니다. rng.choice(풀, size=개수, replace=False)는 풀에서 "중복 없이" 개수만큼 무작위로 뽑습니다.
        tags = list(rng.choice(TAGS_POOL, size=k_tags, replace=False)) if k_tags > 0 else []
        title = f"{company} {year} {doc_type}"
        # doc_type이 "계약서"면 손해배상 관련 문구 중 하나를, 아니면 GENERIC_PHRASES에서 그 doc_type에
        # 맞는 문구를 꺼냅니다. dict.get(키, 기본값)은 키가 없어도 에러 대신 기본값을 돌려줍니다.
        phrase = CONTRACT_PHRASES[rng.integers(0, len(CONTRACT_PHRASES))] if doc_type == "계약서" else GENERIC_PHRASES.get(doc_type, "문서 본문 요약.")
        content = f"{company}의 {year}년 {doc_type}. {phrase}"
        metadata = {"author": author, "company": company, "doc_type": doc_type, "financial_year": year, "pages": pages}
        if tags: metadata["tags"] = tags   # tags가 빈 리스트가 아닐 때만(=태그가 있을 때만) 키를 추가
        # 이 문서의 벡터 = doc_type의 중심 벡터 + 무작위 노이즈. scale=0.3은 노이즈의 표준편차 —
        # 너무 크면 같은 doc_type끼리도 서로 멀어져 군집이 무의미해지고, 너무 작으면 문서들이 거의
        # 똑같아집니다. 만든 뒤 다시 길이 1로 정규화(단위벡터화)합니다.
        emb = centers[dt_idx] + rng.normal(scale=0.3, size=DIM).astype(np.float32)
        emb /= np.linalg.norm(emb)
        # psycopg.types.json.Json(metadata): 딕셔너리를 JSONB 컬럼에 안전하게 넣을 수 있는 형태로 감쌉니다.
        # 튜플의 각 값 순서는 바로 아래 INSERT문의 컬럼 순서(title, content, metadata, ...)와 같아야 합니다.
        rows.append((title, content, psycopg.types.json.Json(metadata), company, doc_type, year, emb))

    # executemany: rows 리스트의 각 튜플에 대해 같은 INSERT문을 반복 실행 — n_add번 execute를
    # 따로 부르는 것보다 한 번에 묶어 보내는 것이 훨씬 빠릅니다. %s가 7개, 튜플 값도 7개로 순서대로 매칭됩니다.
    with conn.cursor() as cur:
        cur.executemany("""INSERT INTO documents (title, content, metadata, company, doc_type, financial_year, embedding)
                             VALUES (%s, %s, %s, %s, %s, %s, %s)""", rows)
    conn.commit()
    print(f"{n_add}건 추가 생성 -> 총 {conn.execute('SELECT count(*) FROM documents').fetchone()[0]}건")

296건 추가 생성 -> 총 300건


### R2 · 인덱스 4종 — 정형(B-tree) · JSONB(GIN) · 벡터(HNSW) · 전문검색(GIN tsvector)

In [6]:
# content_tsv 생성 컬럼 (title + content 기반, STORED)
# 📖 문법 참고: GENERATED ALWAYS AS (수식) STORED는 "생성 컬럼"입니다 — 값을 직접 INSERT/UPDATE로
#    넣는 게 아니라, 지정한 수식으로 다른 컬럼 값을 계산해 항상 자동으로 채워집니다. STORED는 그
#    계산 결과를 매번 다시 계산하지 않고 디스크에 실제로 저장해둔다는 뜻(조회할 때 더 빠름)입니다.
# to_tsvector('simple', 문자열): 문자열을 전문검색(full-text search)에 쓸 수 있는 "tsvector"라는
#    특수 형태로 변환합니다. 'simple'은 언어별 형태소 분석 없이 단순히 토큰(단어)만 나눈다는 설정입니다.
# ||: SQL에서 문자열을 이어 붙이는(연결) 연산자입니다. coalesce(title,'')는 title이 NULL이면 빈 문자열로
#    바꿔서, NULL과 이어붙이면 전체가 NULL이 되어버리는 문제를 막습니다(문자열 + NULL = NULL이기 때문).
conn.execute("""
ALTER TABLE documents ADD COLUMN IF NOT EXISTS content_tsv tsvector
    GENERATED ALWAYS AS (to_tsvector('simple', coalesce(title,'') || ' ' || coalesce(content,''))) STORED;
""")
conn.commit()

# 아래 4개 CREATE INDEX가 이 실습의 핵심 — 서로 다른 4가지 "검색 방식"에 맞는 인덱스 종류를 각각 만듭니다.
# ① (doc_type, financial_year): 일반적인 B-tree 인덱스 — "=" 조건이나 정렬에 강함(정형 컬럼 조합 필터).
# ② USING GIN (metadata jsonb_path_ops): JSONB 전용 GIN 인덱스. jsonb_path_ops는 기본(jsonb_ops)보다
#    크기가 작고, "@>"(포함 여부) 조건 검색에 특화되어 있습니다(뒤 보너스에서 사용).
conn.execute("CREATE INDEX IF NOT EXISTS idx_documents_structured ON documents (doc_type, financial_year);")
conn.execute("CREATE INDEX IF NOT EXISTS idx_documents_metadata_gin ON documents USING GIN (metadata jsonb_path_ops);")
# ③ USING hnsw (embedding vector_cosine_ops) WITH (m=16, ef_construction=64): pgvector가 제공하는
#    HNSW(계층적 그래프 기반) 벡터 인덱스입니다. vector_cosine_ops는 "코사인 거리" 기준으로 비교하겠다는
#    뜻이고, m·ef_construction은 그래프를 얼마나 촘촘하게 만들지 조절하는 값입니다(값이 클수록 정확하지만
#    인덱스 생성이 느려짐 — 여기서는 흔히 쓰는 기본값 정도를 그대로 사용).
conn.execute("CREATE INDEX IF NOT EXISTS idx_documents_embedding_hnsw ON documents USING hnsw (embedding vector_cosine_ops) WITH (m=16, ef_construction=64);")
# ④ USING GIN (content_tsv): 방금 만든 tsvector 컬럼을 위한 전문검색 인덱스.
conn.execute("CREATE INDEX IF NOT EXISTS idx_documents_tsv_gin ON documents USING GIN (content_tsv);")
conn.commit()
# ANALYZE documents: 이 테이블의 데이터 분포(값의 개수·분포 등) 통계를 다시 수집합니다. PostgreSQL의
# 쿼리 옵티마이저는 이 통계를 바탕으로 "인덱스를 쓸지 말지"를 판단하므로, 데이터를 많이 바꾼 뒤에는
# ANALYZE를 실행해줘야 최신 통계로 올바른 판단을 합니다(뒤 실측 셀에서 바로 이 판단을 확인합니다).
conn.execute("ANALYZE documents;")
conn.commit()

print("=== pg_indexes ===")
# pg_indexes: 이 데이터베이스에 존재하는 인덱스 목록을 보여주는 PostgreSQL의 시스템 카탈로그(내장 뷰)입니다.
for r in conn.execute("SELECT indexname FROM pg_indexes WHERE tablename='documents' ORDER BY indexname;").fetchall():
    print(" -", r[0])

=== pg_indexes ===
 - documents_company_idx
 - documents_pkey
 - idx_documents_embedding_hnsw
 - idx_documents_metadata_gin
 - idx_documents_structured
 - idx_documents_tsv_gin


> ⚠️ **흔한 실수**: `(metadata->>'financial_year')::SMALLINT`에서 `->`(JSONB 반환)를 캐스팅 대상으로 쓰면 `cannot cast type jsonb to smallint` 오류가 납니다. 반드시 `->>`(텍스트 반환)를 쓰세요. 컬럼을 `TEXT`로 승격하면 `financial_year >= 2024` 같은 범위 비교가 문자열 비교가 되어버리는 것도 같은 계열의 함정입니다.

### R3 · `hybrid_search()` — 인터페이스 계약 그대로 구현

이 환경엔 실제 `psycopg` 연결(`conn`)이 있으므로, 원본 과제의 시그니처를 **단순화하지 않고 그대로** 구현합니다.

In [7]:
# 📖 문법 참고: def 함수명(..., doc_type=None, financial_year=None, k=5): 처럼 매개변수에 기본값을
#    지정해두면, 호출할 때 그 인자를 생략해도 됩니다(예: hybrid_search(conn, qvec)만 써도 동작).
#    함수 본문 맨 위 """ ... """는 이 함수의 설명을 적어두는 문서화 문자열(docstring)입니다.
def hybrid_search(conn, query_vector, doc_type=None, financial_year=None, k=5):
    """
    반환: [{"id": int, "title": str, "doc_type": str, "financial_year": int, "distance": float}, ...]
    필터가 None이면 전체 대상, 지정되면 AND 조건으로 결합합니다.
    """
    # conditions: WHERE에 들어갈 조건들을 하나씩 리스트에 쌓아가는 방식입니다. params는 그 조건들
    # 중 %s 자리에 채울 실제 값들을 같은 순서로 모아둡니다. 필터가 주어졌을 때만 조건과 값을 추가하므로,
    # 결과적으로 "지정된 필터만 AND로 결합"되는 동적 SQL이 만들어집니다.
    conditions = ["embedding IS NOT NULL"]
    params = []
    if doc_type is not None:
        conditions.append("doc_type = %s"); params.append(doc_type)
    if financial_year is not None:
        conditions.append("financial_year = %s"); params.append(financial_year)
    # " AND ".join(conditions): 리스트의 문자열들을 " AND "로 이어붙여 하나의 문자열로 합칩니다.
    # 예: ["embedding IS NOT NULL", "doc_type = %s"] -> "embedding IS NOT NULL AND doc_type = %s"
    where_clause = " AND ".join(conditions)
    # ⚠️ 여기서 f-string으로 끼워 넣는 것은 "조건절 텍스트(where_clause)" 자체이지 사용자 값이 아닙니다
    # — 값은 전부 %s로 남겨두고 아래 execute에서 params 튜플로 안전하게 전달합니다(값을 직접 f-string으로
    # 끼워 넣는 것은 SQL 인젝션 위험이 있어 피해야 합니다 — 자세한 이유는 바로 다음 마크다운 셀 참고).
    # <=>: pgvector가 제공하는 "코사인 거리" 연산자(값이 작을수록 더 비슷함). ::VECTOR(1024)는 파이썬에서
    # 넘긴 값을 VECTOR(1024) 타입으로 캐스팅합니다.
    sql = f"""
        SELECT id, title, doc_type, financial_year,
               embedding <=> %s::VECTOR(1024) AS distance
        FROM documents
        WHERE {where_clause}
        ORDER BY distance
        LIMIT %s;
    """
    cur = conn.cursor()
    # (query_vector, *params, k): 튜플을 만들 때 *params로 params 리스트의 원소들을 하나씩 풀어
    # 끼워 넣습니다(리스트를 통째로 하나의 항목으로 넣는 게 아니라 개별 값들로 펼침) — 그래서 sql 안의
    # %s 개수(벡터 1개 + 필터 조건 수만큼 + LIMIT 1개)와 정확히 맞아떨어집니다.
    cur.execute(sql, (query_vector, *params, k))
    cols = ["id", "title", "doc_type", "financial_year", "distance"]
    # zip(cols, row): 컬럼명 리스트와 값 튜플을 같은 위치끼리 짝지어줍니다. dict(...)로 그 짝을
    # {"id": ..., "title": ...} 같은 딕셔너리로 만들고, 리스트 컴프리헨션으로 모든 행에 대해 반복합니다.
    return [dict(zip(cols, row)) for row in cur.fetchall()]

# 쿼리 벡터: 계약서 문서 하나를 그대로 사용 (자기 자신이 top-1, distance=0.0이어야 함)
qvec = conn.execute("SELECT embedding FROM documents WHERE doc_type='계약서' ORDER BY id LIMIT 1").fetchone()[0]
# register_vector(conn) 덕분에 여기서 꺼낸 embedding 값은 pgvector 전용 Vector 객체입니다.
# .to_list()로 파이썬 리스트로 바꾸고, np.asarray(..., dtype=np.float32)로 다시 numpy 배열로 만듭니다
# (아래 hybrid_search에 넘길 때 numpy 배열 형태가 필요하기 때문).
qvec = np.asarray(qvec.to_list(), dtype=np.float32)

print("=== 필터 없는 기본 검색 ===")
for r in hybrid_search(conn, qvec, k=5): print(r)

print("\n=== 필터 결합 검색 (doc_type='계약서') ===")
for r in hybrid_search(conn, qvec, doc_type="계약서", k=5): print(r)

=== 필터 없는 기본 검색 ===
{'id': 29, 'title': 'KB 2024 계약서', 'doc_type': '계약서', 'financial_year': 2024, 'distance': 0.0}
{'id': 239, 'title': 'LG 2022 계약서', 'doc_type': '계약서', 'financial_year': 2022, 'distance': 0.9003949731117519}
{'id': 183, 'title': 'KB 2024 정관', 'doc_type': '정관', 'financial_year': 2024, 'distance': 0.9100134947887959}
{'id': 127, 'title': 'LG 2025 반기보고서', 'doc_type': '반기보고서', 'financial_year': 2025, 'distance': 0.9260474368929857}
{'id': 63, 'title': '삼성 2023 press', 'doc_type': 'press', 'financial_year': 2023, 'distance': 0.9287639834706811}

=== 필터 결합 검색 (doc_type='계약서') ===
{'id': 29, 'title': 'KB 2024 계약서', 'doc_type': '계약서', 'financial_year': 2024, 'distance': 0.0}
{'id': 239, 'title': 'LG 2022 계약서', 'doc_type': '계약서', 'financial_year': 2022, 'distance': 0.9003949731117519}
{'id': 73, 'title': '삼성 2022 계약서', 'doc_type': '계약서', 'financial_year': 2022, 'distance': 0.9369984436929687}
{'id': 248, 'title': '신한 2025 계약서', 'doc_type': '계약서', 'financial_year': 2025, 'dista

In [8]:
for r in hybrid_search(conn, qvec, financial_year=2022, k=5): print(r)

{'id': 239, 'title': 'LG 2022 계약서', 'doc_type': '계약서', 'financial_year': 2022, 'distance': 0.9003949731117519}
{'id': 73, 'title': '삼성 2022 계약서', 'doc_type': '계약서', 'financial_year': 2022, 'distance': 0.9369984436929687}
{'id': 284, 'title': 'SK 2022 press', 'doc_type': 'press', 'financial_year': 2022, 'distance': 0.9405817573915863}
{'id': 186, 'title': 'LG 2022 계약서', 'doc_type': '계약서', 'financial_year': 2022, 'distance': 0.9428251457020552}
{'id': 268, 'title': 'SK 2022 사업보고서', 'doc_type': '사업보고서', 'financial_year': 2022, 'distance': 0.9453088971788347}


In [9]:
# 함정 방지 확인 1 — 매우 희소한 필터 조합은 0건이 정상 (예외를 던지면 안 됨)
res_empty = hybrid_search(conn, qvec, doc_type="press", financial_year=2099, k=5)
print("존재하지 않는 조합 -> 반환 건수:", len(res_empty), "(빈 리스트가 정상)")

존재하지 않는 조합 -> 반환 건수: 0 (빈 리스트가 정상)


In [10]:
# 함정 방지 확인 2 — psql로 직접 실행한 결과와 hybrid_search() 결과가 정확히 같아야 함
# (%s, "계약서", 2024): SQL 문자열 안의 %s 세 개(벡터·doc_type·financial_year)에 순서대로 채워질 값들
direct = conn.execute("""
    SELECT id, title, doc_type, financial_year, embedding <=> %s::VECTOR(1024) AS distance
    FROM documents WHERE embedding IS NOT NULL AND doc_type=%s AND financial_year=%s
    ORDER BY distance LIMIT 5
""", (qvec, "계약서", 2024)).fetchall()
via_func = hybrid_search(conn, qvec, doc_type="계약서", financial_year=2024, k=5)
# direct는 (id, title, ...) 튜플들의 리스트라 r[0]으로 첫 번째 값(id)을 꺼내고,
# via_func는 hybrid_search가 만든 딕셔너리들의 리스트라 r["id"]로 같은 값을 꺼냅니다 —
# "직접 짠 SQL"과 "함수로 감싼 SQL"이 정말 같은 결과를 내는지 두 가지 접근 방식을 나란히 비교합니다.
print("직접 SQL ids   :", [r[0] for r in direct])
print("hybrid_search ids:", [r["id"] for r in via_func])
assert [r[0] for r in direct] == [r["id"] for r in via_func]   # 두 리스트가 완전히 같아야 통과
print("-> 일치 확인 완료")

직접 SQL ids   : [29, 169, 109, 114, 275]
hybrid_search ids: [29, 169, 109, 114, 275]
-> 일치 확인 완료


> ⚠️ **흔한 실수**: `f"WHERE doc_type = '{doc_type}'"`처럼 문자열 포매팅으로 값을 직접 SQL에 끼워 넣지 마세요. SQL 인젝션에 취약할 뿐 아니라 값에 작은따옴표가 있으면 쿼리가 깨집니다. 위 코드처럼 **테이블·컬럼명은 f-string으로, 값은 반드시 `%s` + `params` 튜플로** 넘겨 psycopg가 안전하게 이스케이프하게 하세요.
>
> 🏭 **실무에서는**: 메타데이터 필터와 벡터 유사도를 결합한 질의는 RAG(검색 증강 생성) 시스템의 핵심 패턴입니다. 필터 없이 벡터 검색만 하면 관련 없는 연도·문서 유형이 섞여 들어옵니다.

---
## 왜 인덱스가 안 쓰이기도 하는가 — 실측

**HNSW는 "정확한 최근접"이 아니라 "근사 최근접"(ANN) 인덱스입니다.** WHERE 조건이 있으면 옵티마이저는 ①B-tree/GIN으로 먼저 좁히고 그 소수만 정확히 거리 계산·정렬하는 **선필터**, ②HNSW로 벡터 거리순 후보를 먼저 뽑고 그중 조건에 맞는 것만 남기는 **후필터** 중 비용이 싼 쪽을 고릅니다. 아래에서 이 테이블(300건) 규모에서 실제로 어느 쪽이 선택되는지 직접 확인합니다.

In [11]:
def explain_query(conn, sql, params):
    cur = conn.cursor()
    # EXPLAIN (ANALYZE, FORMAT JSON) 쿼리: 이 쿼리를 실제로 한 번 실행해보고(ANALYZE), 그 결과
    # 대신 "어떤 방식(Seq Scan/Index Scan/...)으로 실행했는지, 몇 ms 걸렸는지"를 사람이 읽기보다
    # 프로그램이 다루기 쉬운 JSON 구조로 돌려달라는 명령입니다. f-string으로 {sql}(실행할 쿼리 본문)을
    # EXPLAIN 뒤에 끼워 넣고, 값은 그대로 params로 넘깁니다.
    cur.execute(f"EXPLAIN (ANALYZE, FORMAT JSON) {sql}", params)
    # FORMAT JSON 결과는 "행 1개, 컬럼 1개"이고 그 값이 JSON 배열 안에 실행계획 객체 하나가 든 형태라
    # fetchone()[0]으로 그 배열을 꺼내고, [0]을 한 번 더 써서 배열의 첫(유일한) 원소를 꺼냅니다.
    plan = cur.fetchone()[0][0]
    # 📖 문법 참고: 실행계획은 트리 구조입니다(하나의 단계 아래 또 다른 하위 단계들이 중첩). 어느 깊이에
    # "Index Name"이 있을지 미리 알 수 없으므로, 함수가 자기 자신을 다시 호출하는 "재귀 함수"로
    # 트리를 끝까지 파고들며 찾습니다. node.get("Plans", [])는 하위 단계 목록이 없으면 빈 리스트를
    # 돌려줘 for문이 그냥 0번 반복하고 넘어가게 합니다(에러 방지).
    def find_index(node):
        if "Index Name" in node: return node["Index Name"]
        for c in node.get("Plans", []):
            found = find_index(c)
            if found: return found
        return None
    return {"execution_time_ms": plan["Execution Time"], "node_type": plan["Plan"].get("Node Type"), "index_used": find_index(plan["Plan"])}

# HYBRID_SQL: %s 자리가 6개(벡터, doc_type용 2개, financial_year용 2개, LIMIT)인 SQL 템플릿을
# 변수에 미리 담아두고 아래에서 여러 시나리오에 재사용합니다.
# 📖 문법 참고: (%s::varchar IS NULL OR doc_type = %s) 패턴은 "필터를 선택적으로 켜고 끄는" 흔한
#    트릭입니다 — 앞의 %s(캐스팅된 값)가 NULL이면 OR 앞부분이 참이 되어 그 조건 전체가 항상 참
#    (=필터 없음)이 되고, 값이 있으면 뒤의 doc_type = %s로 실제 필터링이 됩니다. 같은 값을 %s
#    자리 두 곳에 각각 넣어줘야 하므로, 호출할 때 파라미터 튜플에 같은 값이 두 번 들어갑니다.
HYBRID_SQL = """
    SELECT id, title, doc_type, financial_year, embedding <=> %s::VECTOR(1024) AS distance
    FROM documents
    WHERE embedding IS NOT NULL
      AND (%s::varchar IS NULL OR doc_type = %s)
      AND (%s::smallint IS NULL OR financial_year = %s)
    ORDER BY distance LIMIT %s;
"""
# scenarios: 딕셔너리를 원소로 갖는 리스트 — "필터 조합 3가지"를 미리 정의해두고 아래 for문에서
# 하나씩 꺼내 같은 코드로 반복 실행합니다(코드 중복 없이 여러 케이스를 한 번에 확인하는 방법).
scenarios = [
    {"label": "필터 없음",             "doc_type": None,       "financial_year": None},
    {"label": "낮은 선택도(다건 매치)",  "doc_type": "사업보고서", "financial_year": None},
    {"label": "높은 선택도(소수 매치)",  "doc_type": "press",    "financial_year": 2025},
]
print(f"documents 총 건수: {conn.execute('SELECT count(*) FROM documents').fetchone()[0]}건\n")
for s in scenarios:
    # s["doc_type"]처럼 딕셔너리는 대괄호 + 키 이름으로 값을 꺼냅니다. 이 조회는 "이 필터 조합에 실제로
    # 몇 건이 매치되는지"를 미리 세어 아래 출력에서 EXPLAIN 결과와 나란히 비교하기 위한 것입니다.
    n_match = conn.execute(
        "SELECT count(*) FROM documents WHERE (%s::varchar IS NULL OR doc_type=%s) AND (%s::smallint IS NULL OR financial_year=%s)",
        (s["doc_type"], s["doc_type"], s["financial_year"], s["financial_year"])
    ).fetchone()[0]
    r = explain_query(conn, HYBRID_SQL, (qvec, s["doc_type"], s["doc_type"], s["financial_year"], s["financial_year"], 5))
    # 📖 문법 참고: f"{값:20s}"·f"{값:3d}"·f"{값:.3f}"는 파이썬 f-string의 "포맷 지정자"입니다 —
    #    20s/10s/26s는 문자열을 그 폭만큼 왼쪽 정렬(부족하면 공백 채움)해서 표처럼 줄을 맞춰주고,
    #    3d는 정수를 3칸 폭으로, .3f는 소수점 셋째 자리까지의 실수로 표시하라는 뜻입니다.
    print(f"{s['label']:20s} (매치 {n_match:3d}건) -> {r['node_type']:10s} 인덱스={str(r['index_used']):26s} {r['execution_time_ms']:.3f}ms")

documents 총 건수: 300건

필터 없음                (매치 300건) -> Limit      인덱스=None                       1.356ms
낮은 선택도(다건 매치)        (매치  39건) -> Limit      인덱스=None                       0.190ms
높은 선택도(소수 매치)        (매치   8건) -> Limit      인덱스=idx_documents_structured   0.064ms


**실측 결과 읽는 법**: 필터가 없거나(300건) 느슨하면(39건, 13%) 옵티마이저는 `Seq Scan` + `Sort`를 고릅니다 — 이 규모에서는 전체를 훑어 정렬하는 게 B-tree나 HNSW를 타는 것보다 싸다고 판단한 것입니다. 반면 필터가 매우 좁으면(8건, 2.7%) `Bitmap Index Scan using idx_documents_structured`로 갈아탑니다. **인덱스를 만들었다고 항상 쓰이는 게 아니라, 통계 기반 비용 추정으로 갈립니다** — `ANALYZE documents;`를 먼저 실행해야 최신 통계로 이 판단이 정확해집니다.

> 💡 **핵심**: 선필터와 후필터의 관계는 좁은 골목과 넓은 도로 중 어디로 갈지 정하는 내비게이션과 같습니다. 목적지 근처 골목(선택적 필터)이 이미 뚫려 있으면 거기부터 좁혀 들어가는 게 빠르고(선필터), 필터가 느슨해 후보가 여전히 많으면 큰 도로(HNSW)를 타고 가다 마지막에 조건을 확인하는 게 낫습니다(후필터).

### 결과 저장 — `output/` 폴더에 근거 남기기

원본 과제의 `verify.py`·`save_report()`는 이 환경에 없으므로, 같은 목적(EXPLAIN 근거를 파일로 남겨 README에 붙여넣기)을 직접 코드 몇 줄로 대신합니다.

In [12]:
import csv               # csv: 표 형태 데이터를 .csv 파일로 읽고 쓰는 파이썬 표준 라이브러리
from pathlib import Path  # Path: 파일·폴더 경로를 문자열 대신 객체로 다루는 표준 라이브러리(운영체제 상관없이 안전)

output_dir = Path("output")
# .mkdir(exist_ok=True): 폴더를 만듭니다. exist_ok=True 덕분에 이미 폴더가 있어도 에러 없이 넘어갑니다.
output_dir.mkdir(exist_ok=True)

# report: 이 노트북의 핵심 결과를 모아 파일로 남길 딕셔너리. "scenarios" 키에는 아래 반복문에서
# 채울 리스트를 미리 빈 리스트로 준비해둡니다.
report = {"scenarios": [], "backfill_mismatch": mismatch, "index_count": 4}
for s in scenarios:
    n_match = conn.execute(
        "SELECT count(*) FROM documents WHERE (%s::varchar IS NULL OR doc_type=%s) AND (%s::smallint IS NULL OR financial_year=%s)",
        (s["doc_type"], s["doc_type"], s["financial_year"], s["financial_year"])
    ).fetchone()[0]
    r = explain_query(conn, HYBRID_SQL, (qvec, s["doc_type"], s["doc_type"], s["financial_year"], s["financial_year"], 5))
    # {"label": ..., "matches": n_match, **r}: 딕셔너리 리터럴 안에서 **r은 r 딕셔너리
    # (execution_time_ms·node_type·index_used를 담고 있음)의 모든 키-값 쌍을 그대로 펼쳐 넣습니다
    # — 즉 새 딕셔너리 = {label, matches} + r의 내용 전부를 합친 것입니다.
    report["scenarios"].append({"label": s["label"], "matches": n_match, **r})

# with open(경로, "w", encoding="utf-8") as f: 파일을 쓰기 모드("w")로 열고, 블록이 끝나면 자동으로
# 닫습니다. output_dir / "explain_report.json"처럼 Path 객체는 "/" 연산자로 경로를 이어붙일 수 있습니다.
# json.dump(값, f, ...): 파이썬 값을 JSON 텍스트로 바꿔 파일에 바로 씁니다(json.dumps는 문자열만 만들고,
# dump는 파일에 직접 쓴다는 차이). ensure_ascii=False는 한글을 \uXXXX로 이스케이프하지 않고 그대로 저장,
# indent=2는 사람이 읽기 좋게 들여쓰기해서 저장하라는 옵션입니다.
with open(output_dir / "explain_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

idx_rows = conn.execute("SELECT indexname, indexdef FROM pg_indexes WHERE tablename='documents' ORDER BY indexname;").fetchall()
# csv.writer(f): 이 파일에 CSV 형식으로 쓸 수 있는 writer 객체를 만듭니다.
# .writerow(리스트): 리스트를 한 줄(헤더)로 씀. .writerows(튜플들의 리스트): 여러 줄을 한 번에 씀.
with open(output_dir / "index_summary.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["indexname", "indexdef"]); w.writerows(idx_rows)

# output_dir.iterdir(): 이 폴더 안의 파일·폴더 목록을 하나씩 돌려줍니다. p.name은 그 항목의 파일명만
# (전체 경로 없이) 꺼내는 속성입니다 — 리스트 컴프리헨션으로 방금 만든 파일들의 이름만 모아 출력합니다.
print("저장 완료:", [p.name for p in output_dir.iterdir()])
print(json.dumps(report, ensure_ascii=False, indent=2))

저장 완료: ['explain_report.json', 'index_summary.csv']
{
  "scenarios": [
    {
      "label": "필터 없음",
      "matches": 300,
      "execution_time_ms": 1.2,
      "node_type": "Limit",
      "index_used": null
    },
    {
      "label": "낮은 선택도(다건 매치)",
      "matches": 39,
      "execution_time_ms": 0.155,
      "node_type": "Limit",
      "index_used": null
    },
    {
      "label": "높은 선택도(소수 매치)",
      "matches": 8,
      "execution_time_ms": 0.055,
      "node_type": "Limit",
      "index_used": "idx_documents_structured"
    }
  ],
  "backfill_mismatch": 0,
  "index_count": 4
}


---
## 도전 과제 — 선택

### 도전 1 · pre-filtering ↔ post-filtering 역전 지점 찾기

**1단계 — B-tree 경로를 강제로 막아도 옵티마이저가 정말 HNSW로 넘어갈까?**

In [13]:
# 실제 2건만 매치하는 조합을 데이터에서 직접 찾음
# 📖 문법 참고: GROUP BY 1,2,3은 SELECT의 첫째·둘째·셋째 컬럼(company, doc_type, financial_year)
#    기준으로 묶으라는 뜻입니다 — 컬럼 이름을 반복해서 안 적어도 되는 "위치 기반" GROUP BY 표기법.
#    HAVING count(*) BETWEEN 2 AND 6: GROUP BY로 묶은 그룹 중 "그룹 안 행 개수가 2~6개인 것만" 남깁니다
#    (WHERE는 묶기 전 개별 행을, HAVING은 묶은 뒤의 그룹 결과를 거른다는 차이가 있습니다).
combo = conn.execute("""
    SELECT company, doc_type, financial_year, count(*) c FROM documents
    WHERE company IS NOT NULL GROUP BY 1,2,3 HAVING count(*) BETWEEN 2 AND 6 ORDER BY c LIMIT 1
""").fetchone()
# combo는 (company, doc_type, financial_year, count) 형태의 튜플 하나 — 이렇게 튜플을 변수 여러 개에
# 한 번에 나눠 담는 것을 "언패킹(unpacking)"이라고 합니다(튜플의 값 개수와 변수 개수가 같아야 함).
company, doc_type_c, fy_c, c = combo
print(f"선택된 조합: company='{company}' AND doc_type='{doc_type_c}' AND financial_year={fy_c}  (실제 {c}건)")

# SET enable_seqscan = off / enable_bitmapscan = off: 이 연결(세션)에서만 적용되는 임시 설정으로,
# 옵티마이저가 "전체 훑기(Seq Scan)"와 "B-tree 인덱스를 비트맵으로 활용하는 방식"을 아예 못 쓰게
# 강제로 막습니다 — 그래도 여전히 B-tree를 고르는지, 아니면 HNSW로 넘어가는지 실험하기 위함입니다.
conn.execute("SET enable_seqscan = off; SET enable_bitmapscan = off;")
cur = conn.cursor()
cur.execute("""EXPLAIN (ANALYZE, FORMAT JSON)
    SELECT id FROM documents WHERE company=%s AND doc_type=%s AND financial_year=%s
    ORDER BY embedding <=> %s::VECTOR(1024) LIMIT 5;""", (company, doc_type_c, fy_c, qvec))
plan = cur.fetchone()[0][0]["Plan"]

# 앞서 explain_query() 안에 정의했던 것과 같은 "실행계획 트리에서 인덱스 이름을 재귀적으로 찾는" 함수를
# 여기서 독립적으로 다시 정의합니다(이 셀만 따로 실행해도 동작하도록).
def find_index_name(node):
    if "Index Name" in node: return node["Index Name"]
    for ch in node.get("Plans", []):
        found = find_index_name(ch)
        if found: return found
    return None

print("Seq/Bitmap Scan을 꺼도 선택된 인덱스:", find_index_name(plan), "(HNSW가 아니라 여전히 B-tree)")
print("반환 건수:", plan.get("Actual Rows"), "/ 실제", c, "건")
# RESET: 위에서 SET으로 바꾼 세션 설정을 PostgreSQL 기본값으로 되돌립니다(이후 다른 셀에 영향 안 주도록).
conn.execute("RESET enable_seqscan; RESET enable_bitmapscan;")

선택된 조합: company='한화' AND doc_type='정관' AND financial_year=2023  (실제 2건)
Seq/Bitmap Scan을 꺼도 선택된 인덱스: idx_documents_structured (HNSW가 아니라 여전히 B-tree)
반환 건수: 2 / 실제 2 건


<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=localhost user=postgres database=course_db) at 0x1e7d34b4a10>

**결과**: `enable_seqscan`·`enable_bitmapscan`을 꺼서 B-tree의 Bitmap 경로를 막아도, 옵티마이저는 여전히 `Index Scan using idx_documents_structured`(B-tree를 순서대로 훑는 일반 Index Scan)를 택하고 HNSW로 가지 않습니다. **선택적 필터를 위한 인덱스가 이미 있다면, 옵티마이저는 좀처럼 순수 HNSW 후필터 경로로 가지 않습니다.**

**2단계 — 그럼 후필터 손실(반환 건수 < LIMIT)은 언제 재현되는가?** 인덱스가 전혀 없는 JSONB 필드(`author`)만으로 필터링하면서 HNSW만 남기고, `ef_search`를 낮춰봅니다.

In [ ]:
# GROUP BY 1 ORDER BY 2 LIMIT 1: 앞서와 같은 위치 기반 표기법 — 1번째 컬럼(author)으로 묶고,
# 2번째 컬럼(count)으로 정렬해 가장 적게 나온(=가장 희소한) author 값 하나만 뽑습니다.
author_counts = conn.execute("""
    SELECT metadata->>'author' a, count(*) FROM documents WHERE metadata->>'author' IS NOT NULL
    GROUP BY 1 ORDER BY 2 LIMIT 1
""").fetchone()
author, true_cnt = author_counts   # (author, count) 튜플을 두 변수로 언패킹
print(f"author='{author}' 실제 매치: {true_cnt}건 (documents 전체 중, 이 필드엔 인덱스가 전혀 없음)\n")

# author는 metadata(JSONB) 안의 값이라 인덱스가 없습니다. enable_seqscan·enable_bitmapscan까지 끄면
# "벡터 정렬(ORDER BY embedding <=> ...)을 만족시킬 방법이 HNSW 인덱스뿐"인 상황을 만들 수 있습니다.
conn.execute("SET enable_seqscan = off; SET enable_bitmapscan = off;")
for ef in [1, 2, 4, 8, 40]:
    # hnsw.ef_search: HNSW 인덱스가 검색할 때 그래프에서 살펴볼 후보 개수를 조절하는 실행 시점 설정입니다.
    # 값이 작을수록 빠르지만 정확도(놓치는 후보)가 떨어지고, 클수록 느리지만 더 정확해집니다.
    # f"SET hnsw.ef_search = {ef};"처럼 SQL 명령 자체를 f-string으로 만들어(값이 아니라 설정 구문이라
    # %s로 못 넘기므로) 매 반복마다 다른 ef 값으로 세션 설정을 바꿉니다.
    conn.execute(f"SET hnsw.ef_search = {ef};")
    cur = conn.cursor()
    cur.execute("""EXPLAIN (ANALYZE, FORMAT JSON)
        SELECT id FROM documents WHERE metadata->>'author'=%s
        ORDER BY embedding <=> %s::VECTOR(1024) LIMIT %s;""", (author, qvec, true_cnt))
    got = cur.fetchone()[0][0]["Plan"].get("Actual Rows")
    # 조건부 문자열 이어붙이기: "  <- 손실 발생!" 문자열을 got < true_cnt(실제보다 적게 반환됐을 때)일
    # 때만 뒤에 덧붙입니다. 아니면 빈 문자열("")을 더해 아무 변화가 없게 합니다.
    print(f"ef_search={ef:3d}, LIMIT={true_cnt} -> 반환 {got}건 / 실제 {true_cnt}건" + ("  <- 손실 발생!" if got < true_cnt else ""))
conn.execute("RESET enable_seqscan; RESET enable_bitmapscan; RESET hnsw.ef_search;")

**결과 읽는 법**: `author` 필드는 어떤 인덱스도 타지 않으므로, `enable_seqscan`·`enable_bitmapscan`을 끄면 벡터 정렬을 만족할 수 있는 경로가 HNSW뿐입니다. HNSW는 `ef_search`개의 후보만 그래프에서 탐색한 뒤 그 안에서 `author` 조건을 걸러내므로, `ef_search`가 작을수록(심지어 기본값 40에서도) 실제 매치보다 훨씬 적은 결과가 돌아옵니다 — **이것이 "후필터가 결과를 놓칠 수 있다"의 실제 재현**입니다. 이 환경의 pgvector 0.6.0은 이 손실을 자동 보정하는 **iterative index scan**(0.8.0 이후 버전에 추가)이 없어 손실이 그대로 드러납니다.

### 도전 2 · tsvector 3중 검색 — BM25로 가는 다리

In [14]:
# hybrid_search()와 거의 같은 구조에, "키워드 전문검색" 조건 하나가 항상 맨 앞에 추가된 버전입니다.
def triple_search(conn, query_vector, keyword, doc_type=None, k=5):
    # 📖 문법 참고: content_tsv @@ plainto_tsquery('simple', %s)에서 @@는 "이 tsvector가 이
    #    검색어와 매치되는가"를 묻는 전문검색 연산자입니다. plainto_tsquery('simple', 문자열)은
    #    사용자가 입력한 평범한 문자열(특수 검색 문법 없이)을 전문검색용 질의(tsquery)로 바꿔줍니다.
    conditions = ["content_tsv @@ plainto_tsquery('simple', %s)"]
    params = [keyword]
    if doc_type is not None:
        conditions.append("doc_type = %s"); params.append(doc_type)
    where_clause = " AND ".join(conditions)
    # ts_rank(tsvector, tsquery): 검색어가 문서 안에서 얼마나 "관련성 높게" 등장하는지 점수(랭크)를
    # 계산하는 함수입니다 — 매치 여부만 보는 @@와 달리, 정렬용 점수를 추가로 제공합니다.
    sql = f"""
        SELECT id, title, doc_type, ts_rank(content_tsv, plainto_tsquery('simple', %s)) AS text_rank,
               embedding <=> %s::VECTOR(1024) AS distance
        FROM documents WHERE {where_clause} ORDER BY distance LIMIT %s;
    """
    cur = conn.cursor()
    # %s 순서 주의: SELECT의 ts_rank용 %s(keyword) -> distance용 %s(query_vector) -> WHERE절의
    # %s들(params: keyword, [doc_type]) -> LIMIT의 %s(k) 순서로 값을 튜플에 나열해야 합니다.
    cur.execute(sql, (keyword, query_vector, *params, k))
    cols = ["id", "title", "doc_type", "text_rank", "distance"]
    return [dict(zip(cols, row)) for row in cur.fetchall()]

print("'손해배상'이 실제로 들어간 '계약서' 문서만, 그 안에서 벡터 거리순:")
for r in triple_search(conn, qvec, keyword="손해배상", doc_type="계약서", k=5): print(r)

cur = conn.cursor()
cur.execute("""EXPLAIN (ANALYZE, FORMAT JSON)
    SELECT id FROM documents WHERE content_tsv @@ plainto_tsquery('simple','손해배상') AND doc_type='계약서'
    ORDER BY embedding <=> %s::VECTOR(1024) LIMIT 5;""", (qvec,))
# show_plan: 실행계획 트리를 위에서부터 들여쓰기하며 전부 출력하는 재귀 함수입니다(find_index류가
# "찾으면 즉시 반환"이었다면, 이 함수는 "전체 구조를 다 보여주는" 용도). depth=0 기본값에서 시작해
# 하위 노드로 내려갈 때마다 depth+1을 넘겨 들여쓰기 칸수("  "*depth)를 늘립니다.
def show_plan(node, depth=0):
    line = "  "*depth + node.get("Node Type","?")
    if "Index Name" in node: line += f" using {node['Index Name']}"
    print(line)
    for c in node.get("Plans", []): show_plan(c, depth+1)
show_plan(cur.fetchone()[0][0]["Plan"])

'손해배상'이 실제로 들어간 '계약서' 문서만, 그 안에서 벡터 거리순:
{'id': 29, 'title': 'KB 2024 계약서', 'doc_type': '계약서', 'text_rank': 0.06079271, 'distance': 0.0}
{'id': 239, 'title': 'LG 2022 계약서', 'doc_type': '계약서', 'text_rank': 0.06079271, 'distance': 0.9003949731117519}
{'id': 73, 'title': '삼성 2022 계약서', 'doc_type': '계약서', 'text_rank': 0.06079271, 'distance': 0.9369984436929687}
{'id': 248, 'title': '신한 2025 계약서', 'doc_type': '계약서', 'text_rank': 0.06079271, 'distance': 0.9389023221193359}
{'id': 186, 'title': 'LG 2022 계약서', 'doc_type': '계약서', 'text_rank': 0.06079271, 'distance': 0.9428251457020552}
Limit
  Sort
    Bitmap Heap Scan
      BitmapAnd
        Bitmap Index Scan using idx_documents_structured
        Bitmap Index Scan using idx_documents_tsv_gin


**실측 결과**: 구조 필터(`doc_type`)와 전문검색(`content_tsv`)이 서로 다른 GIN/B-tree 인덱스인데도, 옵티마이저는 `BitmapAnd`로 **두 인덱스의 결과를 교집합으로 합쳐서** 후보를 좁힙니다 — PostgreSQL이 여러 인덱스를 동시에 활용할 수 있음을 보여주는 실제 사례입니다.

> 🔬 **더 들어가기**: `to_tsvector('simple', ...)`를 썼습니다 — `'korean'` 설정이 없기 때문입니다. `'simple'`은 형태소 분석 없이 공백 기준으로만 토큰을 나눕니다. 실무 한국어 전문검색은 `mecab-ko` 같은 형태소 분석기를 연결합니다.

**이 3중 검색이 Hybrid Retrieval 수업의 예고편입니다.** 여기서는 `tsvector` 매치를 필터처럼만 썼지만, 그 수업에서는 벡터 점수와 텍스트 점수(BM25)를 가중합으로 랭킹에 반영하는 법을 배웁니다.

### 🔰 보너스 · 태그로 필터링하기

In [15]:
# 📖 문법 참고: jsonb_array_elements_text(metadata->'tags')는 "집합을 반환하는 함수"입니다 —
#    metadata->'tags'로 꺼낸 JSONB 배열(예: ["공시","연간"])을 한 문서당 하나의 값이 아니라,
#    배열 원소 하나당 한 행으로 "펼쳐서" 돌려줍니다. FROM 절에 documents와 콤마로 나란히 쓰면,
#    각 문서 행이 자신의 태그 개수만큼 복제되어 (문서, 태그) 조합의 행들이 만들어집니다
#    (태그가 3개인 문서는 3행으로 늘어남 — SQL의 암묵적 조인의 일종입니다).
tag_counts = conn.execute("""
    SELECT tag, count(*) FROM documents, jsonb_array_elements_text(metadata->'tags') AS tag
    GROUP BY tag ORDER BY 2 DESC;
""").fetchall()
print("태그별 건수:", tag_counts)

MY_TAG = tag_counts[0][0]   # 가장 많이 쓰인 태그 (내림차순 정렬된 결과의 첫 행, 첫 컬럼)
# metadata @> %s::jsonb: "@>"는 왼쪽 JSONB가 오른쪽 JSONB를 포함하는지 확인하는 연산자입니다.
# {"tags": [MY_TAG]}를 포함 조건으로 주면 "metadata의 tags 배열 안에 MY_TAG가 들어있는 문서"를
# 찾는 뜻이 됩니다. json.dumps(...)로 파이썬 딕셔너리를 JSON 문자열로 바꾼 뒤 ::jsonb로 캐스팅합니다
# (파이썬 딕셔너리를 그대로 %s에 넣을 수는 없어 문자열로 변환해서 넘깁니다).
result = conn.execute(
    "SELECT id, title, metadata->'tags' AS tags FROM documents WHERE metadata @> %s::jsonb LIMIT 5;",
    (json.dumps({"tags": [MY_TAG]}),)
).fetchall()
print(f"\ntag='{MY_TAG}' 결과 샘플:", result)

cur = conn.cursor()
cur.execute("EXPLAIN (ANALYZE, FORMAT JSON) SELECT id FROM documents WHERE metadata @> %s::jsonb;", (json.dumps({"tags": [MY_TAG]}),))
plan = cur.fetchone()[0][0]["Plan"]
# 'Index Name' in plan: 이 실행계획 노드에 "Index Name" 키가 있는지 확인 — 있으면 그 인덱스를 실제로
# 사용했다는 뜻이고, 없으면(=Seq Scan 등) 인덱스를 안 탄 것입니다. 조건부 f-string으로 두 경우를 구분해 출력합니다.
print(f"\nEXPLAIN -> {plan['Node Type']}" + (f" using {plan.get('Index Name')}" if 'Index Name' in plan else " (인덱스 미사용)"))

태그별 건수: [('공시', 51), ('긴급', 42), ('대외비', 35), ('연간', 32), ('초안', 30), ('최종본', 30), ('분기', 29), ('검토필요', 29), ('반기', 22)]

tag='공시' 결과 샘플: [(1, '삼성 사업보고서', ['공시', '연간']), (3, '현대 반기보고서', ['공시', '반기']), (16, '한화 2023 이사회의사록', ['공시']), (27, '신한 2022 사업보고서', ['공시', '긴급']), (29, 'KB 2024 계약서', ['공시'])]

EXPLAIN -> Seq Scan (인덱스 미사용)


GIN `jsonb_path_ops` 인덱스가 실제로 쓰이는지는 태그의 선택도에 달려 있습니다 — 이 규모(300건)에서 상위 빈도 태그는 `Seq Scan`이 나올 수 있습니다(위 CP5의 "낮은 선택도" 사례와 같은 이유). 더 희소한 태그나 더 많은 행에서는 인덱스 사용이 뚜렷해집니다.

---
## 마무리 — 자가 체크

- [x] `metadata`의 값을 실제 컬럼(`company`·`doc_type`·`financial_year`)으로 승격하고, 원본과 100% 일치함을 검증했다 (mismatch = 0)
- [x] B-tree·GIN(jsonb_path_ops)·**HNSW(실제 벡터 인덱스)**·GIN(tsvector) **4종 모두 실제로** 만들었다
- [x] `hybrid_search(conn, query_vector, doc_type, financial_year, k)` 함수가 원본 인터페이스 그대로 동작한다
- [x] 선필터·후필터의 EXPLAIN ANALYZE 차이(속도, 결과 누락 가능성)를 직접 실측했다
- [x] 구조 필터 + 전문 검색 + 벡터 유사도를 한 쿼리(`triple_search`)에서 결합했다
- [x] 이 노트북에서 실제 기능과 합성 데이터로 대체한 부분(벡터 값 자체)을 구분해서 설명할 수 있다